# 03 - Bootstrap UQ: composing CIs across analyses

Every analysis in midas_defect returns an `AnalysisResult` with the full ``bootstrap_samples``
array. This is the lever for downstream UQ composition: if you want a confidence interval on a
non-trivial functional of multiple analyses, you operate on the per-bootstrap samples, not on the
summary CIs.

Three patterns shown here:
1. Single-analysis recompute with a custom `stat_fn` (mean instead of median).
2. Cross-analysis composition (matrix/twin ratio CI from the per-sample paired ratio).
3. Tuning `n_boot` and `boot_unit` for cost/precision trade-offs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from midas_defect.bootstrap import bootstrap_population_stat, bootstrapped, percentiles_with_nans
from midas_defect.types import AnalysisResult, BootUnit
rng = np.random.default_rng(0)
per_grain_rho = rng.lognormal(np.log(1.3e13), 0.5, size=250)  # planted Cu-like distribution

## 1. Custom stat_fn

In [ ]:
boot_mean, est_mean, ci_mean   = bootstrap_population_stat(per_grain_rho, stat_fn=np.nanmean,   n_boot=2000)
boot_med, est_med, ci_med      = bootstrap_population_stat(per_grain_rho, stat_fn=np.nanmedian, n_boot=2000)
boot_p90, est_p90, ci_p90      = bootstrap_population_stat(
    per_grain_rho, stat_fn=lambda x: np.nanpercentile(x, 90), n_boot=2000
)
print(f'mean   = {est_mean:.3e}  ({ci_mean[0]:.3e}, {ci_mean[1]:.3e})')
print(f'median = {est_med:.3e}  ({ci_med[0]:.3e}, {ci_med[1]:.3e})')
print(f'p90    = {est_p90:.3e}  ({ci_p90[0]:.3e}, {ci_p90[1]:.3e})')

## 2. Cross-analysis composition: matrix/twin density ratio

In [ ]:
rho_m = rng.lognormal(np.log(7.7e12), 0.4, size=126)
rho_t = rng.lognormal(np.log(4.0e12), 0.4, size=126)
boot_m, _, _ = bootstrap_population_stat(rho_m, n_boot=2000, rng_seed=1)
boot_t, _, _ = bootstrap_population_stat(rho_t, n_boot=2000, rng_seed=2)
ratio_samples = boot_m / boot_t       # paired across the 2000 bootstrap draws
p16, p50, p84 = percentiles_with_nans(ratio_samples, p=(16, 50, 84))
print(f'ratio rho_matrix / rho_twin = {p50:.2f}  ({p16:.2f}, {p84:.2f})')

## 3. @bootstrapped decorator + tuning n_boot

In [ ]:
@bootstrapped(BootUnit.GRAIN, name='rho_demo', units='m^-2')
def rho_compute(values):
    return np.asarray(values, dtype=float)

for n in (50, 200, 1000, 5000):
    r = rho_compute(per_grain_rho, n_boot=n, rng_seed=0)
    width = r.population_ci[1] - r.population_ci[0]
    print(f'n_boot={n:>4d}  median={r.population_median:.3e}  CI width={width:.3e}')